# Stage 3 — Plan A: Temporal Heterogeneous Hypergraph (IEMOCAP)

## Architecture Summary
**5 nodes per utterance**: text, audio, vis_begin, vis_mid, vis_end  
**Visual feature**: OpenFace-AU(8) ⊕ SigLIP2(1152) → (3, 1160) per utterance  
**4 hyperedge types**:
- Type 0 — Multimodal-utterance: {t_i, a_i, vb_i, vm_i, ve_i}
- Type 1 — Visual-arc: {vb_i, vm_i, ve_i} (onset→apex→offset)
- Type 2 — Contextual (5 modality streams, windowed)
- Type 3 — Speaker (all same-speaker nodes per modality, within dialog)

**Propagation**: Two-level attention (node→edge, edge→node), alternating inter/intra schedule (K=4 layers)  
**Loss**: CB-Focal + λ·BCL  
**Evaluation**: 5-fold Leave-One-Session-Out (LOSO), 6 emotions (ang/exc/fru/hap/neu/sad)

## Ablation notes (ablation_iemocap.py, fold1+fold5)
- K=4 outperforms K=2 (0.6224 vs ~0.61 projected) — model needs capacity for multimodal patterns
- Speaker edges help (removing them increases overfitting gap from 0.11 to 0.21)
- BCL detach and intra-first ordering do not improve results
- Val loss rise is 89% BCL inflation, not classification degradation — early stop on WF1 is correct

## Outputs
Saved to `Dataset/Processed/IEMOCAP/stage3_results/`:
- `fold{1-5}_best_model.pt` — checkpoints per fold
- `final_results.json` — WF1/UAF1/ACC mean±std
- `full_results.json` — histories + config
- `training_curves.png`, `confusion_matrices.png`, `per_class_f1.png`
- `tsne_representations.png`, `arc_attention_heatmap.png`, `cross_fold_summary.png`


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json, warnings, time, copy, math
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


In [ ]:
# ============================================================
#  CONFIG  —  tune these before running
# ============================================================
PROJECT_ROOT = Path('/mnt/Work/ML/Thesis/BMVC/Hopeful')
FEAT_DIR     = PROJECT_ROOT / 'Dataset/Processed/IEMOCAP/features'
RESULTS_DIR  = PROJECT_ROOT / 'Dataset/Processed/IEMOCAP/stage3_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IEMOCAP_EMOTIONS = ['ang', 'exc', 'fru', 'hap', 'neu', 'sad']
EMO2IDX          = {e: i for i, e in enumerate(IEMOCAP_EMOTIONS)}
NUM_CLASSES      = 6

# Input feature dimensions
D_TEXT   = 1024
D_AUDIO  = 1024
D_VIS    = 1160   # AU(8) + SigLIP2(1152)

# Architecture — K=4 retained: ablation showed K=2 hurts WF1 despite smaller gap
HIDDEN   = 256
K_LAYERS = 4
W_PAST   = 4
W_FUTURE = 1
DROPOUT  = 0.3
CHUNK    = 64

# Training
EPOCHS        = 60
LR            = 2e-4
WEIGHT_DECAY  = 1e-4
LAMBDA_BCL    = 0.5
BETA_CB       = 0.9999
GAMMA_FOCAL   = 2.0
PATIENCE      = 12
WARMUP_EPOCHS = 5

print(f'Config: HIDDEN={HIDDEN}, K_LAYERS={K_LAYERS}, W_PAST={W_PAST}, W_FUTURE={W_FUTURE}, DROPOUT={DROPOUT}')
print(f'Results dir: {RESULTS_DIR}')


In [3]:
# ============================================================
#  Load pre-extracted features
# ============================================================
print('Loading IEMOCAP features...')
feats_text   = torch.load(FEAT_DIR / 'text_roberta_large.pt',          weights_only=False)
feats_audio  = torch.load(FEAT_DIR / 'audio_microsoft_wavlm_large.pt', weights_only=False)
feats_siglip = torch.load(FEAT_DIR / 'video_siglip2_temporal.pt',      weights_only=False)
feats_au     = torch.load(FEAT_DIR / 'video_openface_au.pt',           weights_only=False)

# Visual absent: both AU and SigLIP2 are all-zero (no face detected in any frame)
visual_absent_map = {
    uid: (float(np.asarray(feats_au[uid]).sum()) == 0.0 and
          float(np.asarray(feats_siglip[uid]).sum()) == 0.0)
    for uid in feats_text
}
n_absent = sum(visual_absent_map.values())

# Load labels, filter to 6 target emotions
labels_df = pd.read_csv(PROJECT_ROOT / 'Dataset/Processed/IEMOCAP/labels.csv')
labels_df = labels_df[labels_df['emotion'].isin(IEMOCAP_EMOTIONS)].copy().reset_index(drop=True)
labels_df['label'] = labels_df['emotion'].map(EMO2IDX)

print(f'Text features loaded: {len(feats_text)} utterances')
print(f'Filtered (6 emotions): {len(labels_df)} utterances')
print(f'Zero-visual utterances: {n_absent}')
print(f'Sessions: {sorted(labels_df["session"].unique())}')
print('\nEmotion distribution:')
print(labels_df['emotion'].value_counts())

Loading IEMOCAP features...
Text features loaded: 10039 utterances
Filtered (6 emotions): 7380 utterances
Zero-visual utterances: 79
Sessions: ['Session1', 'Session2', 'Session3', 'Session4', 'Session5']

Emotion distribution:
emotion
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
Name: count, dtype: int64


In [ ]:
# ============================================================
#  Hypergraph incidence matrix constructor
# ============================================================
def build_incidence_matrix(N, speakers, w_past=W_PAST, w_future=W_FUTURE):
    '''
    Build hypergraph incidence H: (5N, E) and edge_types: (E,).

    Node layout (5N total):
      text  [0,   N)   t_0 .. t_{N-1}
      audio [N,   2N)  a_0 .. a_{N-1}
      vis_b [2N,  3N)  vb_0 .. vb_{N-1}
      vis_m [3N,  4N)  vm_0 .. vm_{N-1}
      vis_e [4N,  5N)  ve_0 .. ve_{N-1}

    Hyperedge layout (7N + 5K edges, 4 edge types):
      MM   [0,   N)    N multimodal-utterance   type 0
      VA   [N,   2N)   N visual-arc             type 1
      CT   [2N,  3N)   N contextual-text        type 2
      CA   [3N,  4N)   N contextual-audio       type 2
      CVb  [4N,  5N)   N contextual-vis-begin   type 2
      CVm  [5N,  6N)   N contextual-vis-mid     type 2
      CVe  [6N,  7N)   N contextual-vis-end     type 2
      SPK  [7N,  7N+5K) 5K speaker edges       type 3
    '''
    unique_spk = list(dict.fromkeys(speakers))
    K       = len(unique_spk)
    spk2idx = {s: i for i, s in enumerate(unique_spk)}
    E       = 7 * N + 5 * K

    rows, cols = [], []
    modality_offsets = [0, N, 2*N, 3*N, 4*N]

    for i in range(N):
        # MM hyperedge (edge index i)
        for node in [i, N+i, 2*N+i, 3*N+i, 4*N+i]:
            rows.append(node); cols.append(i)

        # Visual-arc hyperedge (edge index N+i)
        for node in [2*N+i, 3*N+i, 4*N+i]:
            rows.append(node); cols.append(N + i)

        # Contextual hyperedges: 5 modality streams (edge indices 2N + m*N + i)
        window = range(max(0, i - w_past), min(N, i + w_future + 1))
        for m_idx, offset in enumerate(modality_offsets):
            e_idx = 2*N + m_idx*N + i
            for j in window:
                rows.append(offset + j); cols.append(e_idx)

        # Speaker hyperedges (edge index 7N + k*5 + m_idx)
        k = spk2idx[speakers[i]]
        for m_idx, offset in enumerate(modality_offsets):
            rows.append(offset + i); cols.append(7*N + k*5 + m_idx)

    V      = 5 * N
    rows_t = torch.tensor(rows, dtype=torch.long)
    cols_t = torch.tensor(cols, dtype=torch.long)
    vals   = torch.ones(len(rows_t), dtype=torch.float32)
    H = torch.sparse_coo_tensor(
        torch.stack([rows_t, cols_t]), vals, (V, E)
    ).coalesce().to_dense()

    edge_types = torch.cat([
        torch.zeros(N,    dtype=torch.long),        # MM
        torch.ones(N,     dtype=torch.long),         # VA
        torch.full((5*N,), 2, dtype=torch.long),     # Contextual
        torch.full((5*K,), 3, dtype=torch.long),     # Speaker
    ])

    assert H.shape == (V, E) and edge_types.shape[0] == E
    return H, edge_types


# Verification
N_v, spk_v = 10, ['F','M','F','M','F','M','F','M','F','M']
H_v, et_v  = build_incidence_matrix(N_v, spk_v)
print(f'H shape:   {H_v.shape}  (expected {5*N_v}, {7*N_v + 5*2})')
print(f'Edge types: {et_v.bincount().tolist()}  (MM, VA, Ctx, Spk)')
mm0 = H_v[:, 0].nonzero().squeeze().tolist()
va0 = H_v[:, N_v].nonzero().squeeze().tolist()
print(f'MM edge 0 members (expect [0,10,20,30,40]): {mm0}')
print(f'VA edge {N_v} members (expect [20,30,40]): {va0}')
print('Incidence matrix OK.')


In [5]:
# ============================================================
#  Dialog feature builder
# ============================================================
def build_dialog_features(dialog_id, dialog_df):
    '''
    Build feature tensors + incidence matrix for one IEMOCAP dialog.
    dialog_df: rows for this dialog_id, sorted by start_time.
    Returns dict with keys: dialog_id, utts, text, audio, vis,
                            vis_absent, labels, speakers, N, H_mat, et
    '''
    sub  = dialog_df[dialog_df['dialog'] == dialog_id].sort_values('start_time').reset_index(drop=True)
    utts = sub['utt_id'].tolist()
    N    = len(utts)

    text_feat = torch.from_numpy(
        np.stack([np.asarray(feats_text[u],  dtype=np.float32) for u in utts]))

    audio_feat = torch.from_numpy(
        np.stack([np.asarray(feats_audio[u], dtype=np.float32) for u in utts]))

    vis_list = []
    for u in utts:
        au  = np.asarray(feats_au[u],     dtype=np.float32)  # (3, 8)
        sig = np.asarray(feats_siglip[u], dtype=np.float32)  # (3, 1152)
        vis_list.append(np.concatenate([au, sig], axis=-1))  # (3, 1160)
    vis_feat = torch.from_numpy(np.stack(vis_list))

    vis_absent = torch.tensor([visual_absent_map[u] for u in utts], dtype=torch.bool)
    labels_t   = torch.tensor(sub['label'].tolist(), dtype=torch.long)
    speakers   = sub['speaker'].tolist()

    # Pre-build incidence matrix (cached — reused every epoch)
    H_mat, et = build_incidence_matrix(N, speakers)

    return dict(
        dialog_id  = dialog_id,
        utts       = utts,
        text       = text_feat,
        audio      = audio_feat,
        vis        = vis_feat,
        vis_absent = vis_absent,
        labels     = labels_t,
        speakers   = speakers,
        N          = N,
        H_mat      = H_mat,   # (5N, E) float32, CPU
        et         = et,      # (E,) long, CPU
    )


# Quick shape check
sample_dia = labels_df['dialog'].iloc[0]
sample     = build_dialog_features(sample_dia, labels_df)
print(f'Sample dialog "{sample_dia}":  N={sample["N"]}')
print(f'  text  : {sample["text"].shape}')
print(f'  audio : {sample["audio"].shape}')
print(f'  vis   : {sample["vis"].shape}')
print(f'  labels: {sample["labels"].shape}')
print(f'  H_mat : {sample["H_mat"].shape}')
print(f'  et    : {sample["et"].shape}')

Sample dialog "Ses01F_impro01":  N=26
  text  : torch.Size([26, 1024])
  audio : torch.Size([26, 1024])
  vis   : torch.Size([26, 3, 1160])
  labels: torch.Size([26])
  H_mat : torch.Size([130, 192])
  et    : torch.Size([192])


In [6]:
# ============================================================
#  Model components
# ============================================================

class ModalityProjector(nn.Module):
    '''Linear → LayerNorm → GELU → Dropout projection.'''
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class VisualProjector(nn.Module):
    '''Projects (N, 3, 1160) -> (N, 3, d).
    Absent utterances (no face detected) replaced with learned embedding.
    '''
    def __init__(self, vis_in=1160, d=256, dropout=0.3):
        super().__init__()
        self.proj               = ModalityProjector(vis_in, d, dropout)
        self.visual_absent_embed = nn.Parameter(torch.zeros(3, d))

    def forward(self, vis_feat, vis_absent):
        '''
        vis_feat  : (N, 3, 1160)
        vis_absent: (N,) bool
        Returns   : (N, 3, d)
        '''
        h = self.proj(vis_feat)                                        # (N, 3, d)
        if vis_absent.any():
            absent_fill = self.visual_absent_embed.unsqueeze(0)        # (1, 3, d)
            mask        = vis_absent.view(-1, 1, 1).expand_as(h)       # (N, 3, d)
            h = torch.where(mask, absent_fill.expand_as(h), h)
        return h


class HypergraphConvLayer(nn.Module):
    '''Two-level attention hypergraph convolution.

    Step 1 (Node→Hyperedge): soft attention over member nodes per edge.
    Step 2 (Hyperedge→Node): soft attention over incident edges per node,
                              conditioned on edge-type embedding. Chunked
                              over nodes to bound peak VRAM usage.
    '''
    def __init__(self, d, num_edge_types=4, dropout=0.3, chunk=64):
        super().__init__()
        self.d     = d
        self.chunk = chunk

        self.node_attn      = nn.Linear(d, 1)
        self.edge_type_emb  = nn.Embedding(num_edge_types, d)
        self.hedge_attn     = nn.Linear(2 * d, 1)
        self.W_update       = nn.Linear(d, d)
        self.norm           = nn.LayerNorm(d)
        self.dropout        = nn.Dropout(dropout)

    def forward(self, X, H, edge_types):
        '''
        X          : (V, d)   node features
        H          : (V, E)   incidence matrix (float 0/1)
        edge_types : (E,)     long, values in [0, num_edge_types)
        Returns    : (V, d)
        '''
        V, d = X.shape
        E    = H.shape[1]

        # Step 1: Node -> Hyperedge
        # Compute per-node global importance score, softmax over member nodes per edge
        node_scores = self.node_attn(X).expand(-1, E)          # (V, E)
        node_scores = node_scores.masked_fill(H == 0, -1e9)
        alpha       = F.softmax(node_scores, dim=0) * H        # (V, E)
        B           = alpha.T @ X                               # (E, d) edge messages

        # Step 2: Hyperedge -> Node (chunked to limit peak VRAM)
        type_emb       = self.edge_type_emb(edge_types)        # (E, d)
        B_typed        = B + type_emb                           # (E, d)
        hedge_attn_mat = torch.zeros(V, E, device=X.device)

        for start in range(0, V, self.chunk):
            end      = min(start + self.chunk, V)
            C        = end - start
            x_chunk  = X[start:end]                             # (C, d)
            x_exp    = x_chunk.unsqueeze(1).expand(-1, E, -1)  # (C, E, d)
            b_exp    = B_typed.unsqueeze(0).expand(C, -1, -1)  # (C, E, d)
            inp      = torch.cat([x_exp, b_exp], dim=-1)        # (C, E, 2d)
            scores   = self.hedge_attn(inp).squeeze(-1)         # (C, E)
            H_chunk  = H[start:end]                             # (C, E)
            scores   = scores.masked_fill(H_chunk == 0, -1e9)
            hedge_attn_mat[start:end] = F.softmax(scores, dim=1) * H_chunk

        M     = hedge_attn_mat @ B                              # (V, d)
        X_new = self.norm(X + self.dropout(F.gelu(self.W_update(M))))
        return X_new


class PlanAModel(nn.Module):
    '''Temporal Heterogeneous Hypergraph for MERC (Plan A).

    5 nodes per utterance. Alternating inter/intra-modal propagation.
    Arc attention pool over visual nodes. CB-Focal + BCL training loss.
    '''
    def __init__(self, d=256, num_classes=6, k_layers=4,
                 dropout=0.3, w_past=4, w_future=1, chunk=CHUNK):
        super().__init__()
        self.d        = d
        self.k_layers = k_layers
        self.w_past   = w_past
        self.w_future = w_future

        self.text_proj  = ModalityProjector(D_TEXT,  d, dropout)
        self.audio_proj = ModalityProjector(D_AUDIO, d, dropout)
        self.vis_proj   = VisualProjector(D_VIS, d, dropout)

        self.layers = nn.ModuleList([
            HypergraphConvLayer(d, num_edge_types=4, dropout=dropout, chunk=chunk)
            for _ in range(k_layers)
        ])

        self.arc_attn  = nn.Linear(d, 1)

        self.fusion_mlp = nn.Sequential(
            nn.Linear(3 * d, d),
            nn.LayerNorm(d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, num_classes),
        )

        self.bcl_head = nn.Sequential(
            nn.Linear(3 * d, 256),
            nn.GELU(),
            nn.Linear(256, 128),
        )

    def forward(self, text, audio, vis, vis_absent, H_mat, edge_types, return_feats=False):
        '''
        text       : (N, 1024)
        audio      : (N, 1024)
        vis        : (N, 3, 1160)
        vis_absent : (N,) bool
        H_mat      : (5N, E) float  — pre-built incidence matrix (moved to device here)
        edge_types : (E,)  long    — edge type ids
        Returns    : logits (N, C) [, feats (N, 128), arc_weights (N, 3)]
        '''
        N = text.shape[0]

        # Project to hidden dim
        ht = self.text_proj(text)                         # (N, d)
        ha = self.audio_proj(audio)                       # (N, d)
        hv = self.vis_proj(vis, vis_absent)               # (N, 3, d)

        # Flatten to 5N node feature matrix
        X = torch.cat([ht, ha, hv[:, 0], hv[:, 1], hv[:, 2]], dim=0)  # (5N, d)

        # Move incidence matrix to device
        H  = H_mat.to(X.device)
        ET = edge_types.to(X.device)

        # Pre-split for alternating schedule
        intra_mask = (ET == 1) | (ET == 2)   # VA + Contextual
        inter_mask = (ET == 0) | (ET == 3)   # MM + Speaker
        H_intra = H[:, intra_mask];  ET_intra = ET[intra_mask]
        H_inter = H[:, inter_mask];  ET_inter = ET[inter_mask]

        # K layers: even=inter-modal, odd=intra-modal
        for li, layer in enumerate(self.layers):
            if li % 2 == 0:
                X = layer(X, H_inter, ET_inter)
            else:
                X = layer(X, H_intra, ET_intra)

        # Unpack node features
        ht_out  = X[0*N : 1*N]   # (N, d)
        ha_out  = X[1*N : 2*N]
        hvb_out = X[2*N : 3*N]
        hvm_out = X[3*N : 4*N]
        hve_out = X[4*N : 5*N]

        # Attention pool over 3 visual arc nodes
        vis_stack  = torch.stack([hvb_out, hvm_out, hve_out], dim=1)  # (N, 3, d)
        arc_scores = self.arc_attn(vis_stack).squeeze(-1)              # (N, 3)
        arc_weights = F.softmax(arc_scores, dim=-1)                    # (N, 3)
        hv_pooled  = (arc_weights.unsqueeze(-1) * vis_stack).sum(1)   # (N, d)

        h_fused = torch.cat([ht_out, ha_out, hv_pooled], dim=-1)      # (N, 3d)
        logits  = self.fusion_mlp(h_fused)                             # (N, C)

        if return_feats:
            feats = F.normalize(self.bcl_head(h_fused), dim=-1)        # (N, 128)
            return logits, feats, arc_weights
        return logits


# --- Sanity check ---
print('Sanity checking PlanAModel...')
_N  = 8
_sp = ['F', 'M'] * (_N // 2)
_H, _et = build_incidence_matrix(_N, _sp)
_m  = PlanAModel(d=32, num_classes=NUM_CLASSES, k_layers=2, chunk=16).to(DEVICE)
_t  = torch.randn(_N, 1024).to(DEVICE)
_a  = torch.randn(_N, 1024).to(DEVICE)
_v  = torch.randn(_N, 3, 1160).to(DEVICE)
_ab = torch.zeros(_N, dtype=torch.bool).to(DEVICE)
_lg, _ft, _aw = _m(_t, _a, _v, _ab, _H, _et, return_feats=True)
print(f'  logits={_lg.shape}  feats={_ft.shape}  arc={_aw.shape}')
_lg.sum().backward()
print('  Backward: OK')
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'  Params (d=32): {n_params:,}')
del _m, _t, _a, _v, _ab, _lg, _ft, _aw, _H, _et
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('PlanAModel OK.')

Sanity checking PlanAModel...
  logits=torch.Size([8, 6])  feats=torch.Size([8, 128])  arc=torch.Size([8, 3])
  Backward: OK
  Params (d=32): 166,859
PlanAModel OK.


In [7]:
# ============================================================
#  Loss functions
# ============================================================

class CBFocalLoss(nn.Module):
    '''Class-Balanced Focal Loss (Cui et al., CVPR 2019).
    Inverse effective-number weighting + focal modulation.
    '''
    def __init__(self, class_counts, beta=0.9999, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        nc  = torch.tensor(class_counts, dtype=torch.float32)
        eff = (1.0 - beta ** nc) / (1.0 - beta)
        w   = (1.0 - beta) / eff
        w   = w / w.sum() * len(w)   # normalize: sum = C
        self.register_buffer('weights', w)

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        p_t   = torch.exp(-ce)
        focal = (1.0 - p_t) ** self.gamma * ce
        return focal.mean()


class BalancedContrastiveLoss(nn.Module):
    '''Balanced Contrastive Learning (Zhu et al., CVPR 2022).
    Reduces head-class dominance in supervised contrastive learning.
    Features must be L2-normalised before calling (model BCL head does this).
    '''
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        '''
        features : (N, 128) unit-norm
        labels   : (N,) long
        '''
        N = features.shape[0]
        if N < 4 or labels.unique().numel() < 2:
            return torch.tensor(0.0, device=features.device, requires_grad=True)

        sim       = features @ features.T / self.temperature   # (N, N)
        diag      = torch.eye(N, dtype=torch.bool, device=features.device)
        same_cls  = (labels.unsqueeze(0) == labels.unsqueeze(1))  # (N, N)
        pos_mask  = same_cls & ~diag
        n_pos     = pos_mask.sum(1, keepdim=True).float().clamp(min=1)

        log_denom = torch.logsumexp(sim.masked_fill(diag, -1e9), dim=1, keepdim=True)
        log_prob  = sim - log_denom

        loss = -(pos_mask.float() * log_prob / n_pos).sum(1).mean()
        return loss


# Verify
print('Checking loss functions...')
_cc = [1103, 1041, 1849, 595, 1708, 1084]  # IEMOCAP 6-class counts
_cb = CBFocalLoss(_cc)
w_arr = _cb.weights.numpy()
print(f'CB-Focal weights: {dict(zip(IEMOCAP_EMOTIONS, w_arr.round(4)))}')
print(f'  min (fru={_cc[2]}): {w_arr.min():.4f}  max (hap={_cc[3]}): {w_arr.max():.4f}')
_bcl = BalancedContrastiveLoss()
_f   = F.normalize(torch.randn(8, 128), dim=-1)
_l   = torch.tensor([0,0,1,1,2,2,3,3])
print(f'BCL test value: {_bcl(_f, _l).item():.4f}')
del _cb, _bcl, _f, _l
print('Loss functions OK.')

Checking loss functions...
CB-Focal weights: {'ang': 0.9733, 'exc': 1.0281, 'fru': 0.6021, 'hap': 1.7596, 'neu': 0.6474, 'sad': 0.9894}
  min (fru=1849): 0.6021  max (hap=595): 1.7596
BCL test value: 2.1032
Loss functions OK.


In [8]:
# ============================================================
#  Training utilities
# ============================================================

def compute_metrics(all_labels, all_preds):
    wf1     = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    uaf1    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    acc     = accuracy_score(all_labels, all_preds)
    per_cls = f1_score(all_labels, all_preds, average=None,
                       zero_division=0, labels=list(range(NUM_CLASSES)))
    return {'wf1': float(wf1), 'uaf1': float(uaf1), 'acc': float(acc),
            'per_class': per_cls.tolist()}


def evaluate(model, dialogs, crit_focal, crit_bcl):
    '''Run inference on dialogs, return (metrics_dict, all_true, all_preds).'''
    model.eval()
    total_loss, all_preds, all_true = 0.0, [], []
    with torch.no_grad():
        for d in dialogs:
            if d['N'] < 2: continue
            text   = d['text'].to(DEVICE)
            audio  = d['audio'].to(DEVICE)
            vis    = d['vis'].to(DEVICE)
            absent = d['vis_absent'].to(DEVICE)
            labels = d['labels'].to(DEVICE)
            H_mat  = d['H_mat']
            et     = d['et']

            logits, feats, _ = model(text, audio, vis, absent, H_mat, et, return_feats=True)
            loss = crit_focal(logits, labels) + LAMBDA_BCL * crit_bcl(feats, labels)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_true, all_preds)
    metrics['loss'] = total_loss / max(len(dialogs), 1)
    return metrics, all_true, all_preds


def train_fold(model, train_dialogs, val_dialogs, resume_path=None):
    '''Train one LOSO fold. Returns (best_state_dict, history, best_val_wf1).
    resume_path: if given, saves a per-epoch checkpoint there and resumes from it
                 if the file already exists (allows crash recovery).
    '''
    all_train_labels = torch.cat([d['labels'] for d in train_dialogs])
    class_counts     = [(all_train_labels == c).sum().item() for c in range(NUM_CLASSES)]
    print(f'    Class counts: {dict(zip(IEMOCAP_EMOTIONS, class_counts))}')

    crit_focal = CBFocalLoss(class_counts, BETA_CB, GAMMA_FOCAL).to(DEVICE)
    crit_bcl   = BalancedContrastiveLoss(0.07).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)

    history = {'train_loss': [], 'val_loss': [],
               'train_wf1': [], 'val_wf1': []}
    best_wf1, best_state, no_improve = 0.0, None, 0
    start_epoch = 0

    # Resume from per-epoch checkpoint if available
    if resume_path is not None and Path(resume_path).exists():
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        history     = ckpt['history']
        best_wf1    = ckpt['best_wf1']
        best_state  = ckpt['best_state']
        no_improve  = ckpt['no_improve']
        start_epoch = ckpt['epoch'] + 1
        print(f'    Resumed from epoch {start_epoch} (best_wf1={best_wf1:.4f})')

    for epoch in range(start_epoch, EPOCHS):
        # Linear warmup
        if epoch < WARMUP_EPOCHS:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * (epoch + 1) / WARMUP_EPOCHS

        model.train()
        train_loss, train_preds, train_true = 0.0, [], []
        order = list(range(len(train_dialogs)))
        np.random.shuffle(order)

        for di in order:
            d = train_dialogs[di]
            if d['N'] < 2: continue
            text   = d['text'].to(DEVICE)
            audio  = d['audio'].to(DEVICE)
            vis    = d['vis'].to(DEVICE)
            absent = d['vis_absent'].to(DEVICE)
            labels = d['labels'].to(DEVICE)

            optimizer.zero_grad()
            logits, feats, _ = model(
                text, audio, vis, absent, d['H_mat'], d['et'], return_feats=True)
            loss = crit_focal(logits, labels) + LAMBDA_BCL * crit_bcl(feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss  += loss.item()
            train_preds.extend(logits.argmax(-1).detach().cpu().tolist())
            train_true.extend(labels.cpu().tolist())

        if epoch >= WARMUP_EPOCHS:
            scheduler.step()

        val_m, _, _    = evaluate(model, val_dialogs, crit_focal, crit_bcl)
        train_m        = compute_metrics(train_true, train_preds)

        history['train_loss'].append(train_loss / max(len(train_dialogs), 1))
        history['val_loss'].append(val_m['loss'])
        history['train_wf1'].append(train_m['wf1'])
        history['val_wf1'].append(val_m['wf1'])

        if (epoch + 1) % 10 == 0 or epoch < 3:
            print(f'    Epoch {epoch+1:3d}  '
                  f'train_wf1={train_m["wf1"]:.4f}  '
                  f'val_wf1={val_m["wf1"]:.4f}  '
                  f'loss={val_m["loss"]:.4f}')

        if val_m['wf1'] > best_wf1:
            best_wf1   = val_m['wf1']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'    Early stop at epoch {epoch+1}')
                break

        # Save per-epoch resume checkpoint
        if resume_path is not None:
            torch.save({
                'epoch'          : epoch,
                'model_state'    : model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'history'        : history,
                'best_wf1'       : best_wf1,
                'best_state'     : best_state,
                'no_improve'     : no_improve,
            }, resume_path)

    # Remove resume checkpoint on clean completion
    if resume_path is not None and Path(resume_path).exists():
        Path(resume_path).unlink()

    return best_state, history, best_wf1

In [9]:
# ============================================================
#  Pre-build all dialog data grouped by session
# ============================================================
print('Pre-building dialog features (all 5 sessions)...')
t0 = time.time()

SESSIONS = ['Session1', 'Session2', 'Session3', 'Session4', 'Session5']
all_dialogs_by_session = {}

for sess in SESSIONS:
    sess_df  = labels_df[labels_df['session'] == sess]
    dial_ids = sess_df['dialog'].unique()
    dialogs  = []
    for dia_id in dial_ids:
        d = build_dialog_features(dia_id, sess_df)
        if d['N'] >= 2:
            dialogs.append(d)
    all_dialogs_by_session[sess] = dialogs
    n_utt = sum(d['N'] for d in dialogs)
    print(f'  {sess}: {len(dialogs)} dialogs, {n_utt} utterances')

total_dia = sum(len(v) for v in all_dialogs_by_session.values())
total_utt = sum(sum(d['N'] for d in v) for v in all_dialogs_by_session.values())
print(f'\nTotal: {total_dia} dialogs, {total_utt} utterances  '
      f'(elapsed: {time.time()-t0:.1f}s)')

Pre-building dialog features (all 5 sessions)...
  Session1: 28 dialogs, 1365 utterances
  Session2: 30 dialogs, 1348 utterances
  Session3: 32 dialogs, 1533 utterances
  Session4: 30 dialogs, 1512 utterances
  Session5: 31 dialogs, 1622 utterances

Total: 151 dialogs, 7380 utterances  (elapsed: 0.3s)


In [10]:
# ============================================================
#  5-Fold LOSO training loop
# ============================================================
fold_results = {}

for fold_idx, test_sess in enumerate(SESSIONS):
    fold_ckpt_path  = RESULTS_DIR / f'fold{fold_idx+1}_best_model.pt'
    resume_path     = RESULTS_DIR / f'resume_fold{fold_idx+1}.pt'

    # Skip fold if final checkpoint already exists (resume after full crash)
    if fold_ckpt_path.exists():
        print(f'\nFold {fold_idx+1}/5 ({test_sess}) — already done, loading saved checkpoint.')
        saved = torch.load(fold_ckpt_path, map_location=DEVICE, weights_only=False)
        # Use config stored in checkpoint, not current CONFIG globals
        cfg = saved.get('config', {'hidden': HIDDEN, 'k_layers': K_LAYERS,
                                    'w_past': W_PAST, 'w_future': W_FUTURE})
        val_dialogs   = all_dialogs_by_session[test_sess]
        train_dialogs = [d for s in SESSIONS if s != test_sess
                           for d in all_dialogs_by_session[s]]
        model = PlanAModel(
            d=cfg['hidden'], num_classes=NUM_CLASSES, k_layers=cfg['k_layers'],
            dropout=DROPOUT, w_past=cfg['w_past'], w_future=cfg['w_future']
        ).to(DEVICE)
        model.load_state_dict(saved['state_dict'])
        all_tl = torch.cat([d['labels'] for d in train_dialogs])
        cc     = [(all_tl == c).sum().item() for c in range(NUM_CLASSES)]
        cf     = CBFocalLoss(cc, BETA_CB, GAMMA_FOCAL).to(DEVICE)
        cb     = BalancedContrastiveLoss(0.07).to(DEVICE)
        final_m, final_true, final_preds = evaluate(model, val_dialogs, cf, cb)
        arc_by_label  = {c: [] for c in range(NUM_CLASSES)}
        reps_list, reps_labels = [], []
        model.eval()
        with torch.no_grad():
            for d in val_dialogs:
                if d['N'] < 2: continue
                _, feats, arc_w = model(
                    d['text'].to(DEVICE), d['audio'].to(DEVICE),
                    d['vis'].to(DEVICE),  d['vis_absent'].to(DEVICE),
                    d['H_mat'], d['et'], return_feats=True)
                reps_list.append(feats.cpu().numpy())
                reps_labels.extend(d['labels'].tolist())
                for c in range(NUM_CLASSES):
                    mask = d['labels'] == c
                    if mask.any():
                        arc_by_label[c].append(arc_w[mask].cpu().numpy())
        fold_results[test_sess] = {
            'metrics'     : saved['metrics'],
            'history'     : saved['history'],
            'true'        : final_true,
            'preds'       : final_preds,
            'arc_by_label': arc_by_label,
            'reps'        : np.concatenate(reps_list, axis=0),
            'reps_labels' : np.array(reps_labels),
        }
        print(f'  WF1={saved["metrics"]["wf1"]:.4f}  UAF1={saved["metrics"]["uaf1"]:.4f}')
        continue

    print(f'\n{"="*60}')
    print(f'Fold {fold_idx+1}/5  |  Test: {test_sess}')

    train_dialogs = [d for s in SESSIONS if s != test_sess
                       for d in all_dialogs_by_session[s]]
    val_dialogs   = all_dialogs_by_session[test_sess]
    print(f'  Train: {len(train_dialogs)} dialogs  |  Val: {len(val_dialogs)} dialogs')

    model = PlanAModel(
        d=HIDDEN, num_classes=NUM_CLASSES, k_layers=K_LAYERS,
        dropout=DROPOUT, w_past=W_PAST, w_future=W_FUTURE
    ).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {n_params:,}')

    t_fold = time.time()
    best_state, history, best_val_wf1 = train_fold(
        model, train_dialogs, val_dialogs, resume_path=resume_path)
    elapsed = time.time() - t_fold

    # Final eval with best weights
    model.load_state_dict(best_state)
    all_tl = torch.cat([d['labels'] for d in train_dialogs])
    cc     = [(all_tl == c).sum().item() for c in range(NUM_CLASSES)]
    cf     = CBFocalLoss(cc, BETA_CB, GAMMA_FOCAL).to(DEVICE)
    cb     = BalancedContrastiveLoss(0.07).to(DEVICE)
    final_m, final_true, final_preds = evaluate(model, val_dialogs, cf, cb)

    # Collect arc weights + BCL reps for visualization
    arc_by_label  = {c: [] for c in range(NUM_CLASSES)}
    reps_list, reps_labels = [], []
    model.eval()
    with torch.no_grad():
        for d in val_dialogs:
            if d['N'] < 2: continue
            _, feats, arc_w = model(
                d['text'].to(DEVICE), d['audio'].to(DEVICE),
                d['vis'].to(DEVICE),  d['vis_absent'].to(DEVICE),
                d['H_mat'], d['et'], return_feats=True)
            reps_list.append(feats.cpu().numpy())
            reps_labels.extend(d['labels'].tolist())
            for c in range(NUM_CLASSES):
                mask = d['labels'] == c
                if mask.any():
                    arc_by_label[c].append(arc_w[mask].cpu().numpy())

    # Save final fold checkpoint
    ckpt = {
        'state_dict'  : best_state,
        'fold'        : fold_idx + 1,
        'test_session': test_sess,
        'metrics'     : final_m,
        'history'     : history,
        'config'      : {'hidden': HIDDEN, 'k_layers': K_LAYERS,
                         'w_past': W_PAST, 'w_future': W_FUTURE},
    }
    torch.save(ckpt, fold_ckpt_path)

    fold_results[test_sess] = {
        'metrics'     : final_m,
        'history'     : history,
        'true'        : final_true,
        'preds'       : final_preds,
        'arc_by_label': arc_by_label,
        'reps'        : np.concatenate(reps_list, axis=0),
        'reps_labels' : np.array(reps_labels),
    }

    print(f'\n  Fold {fold_idx+1} done in {elapsed/60:.1f} min')
    print(f'  WF1={final_m["wf1"]:.4f}  UAF1={final_m["uaf1"]:.4f}  ACC={final_m["acc"]:.4f}')

print(f'\n{"="*60}\nAll 5 folds complete.')


Fold 1/5  |  Test: Session1
  Train: 123 dialogs  |  Val: 28 dialogs
  Params: 1,389,451
    Class counts: {'ang': 874, 'exc': 898, 'fru': 1569, 'hap': 460, 'neu': 1324, 'sad': 890}
    Epoch   1  train_wf1=0.2122  val_wf1=0.2335  loss=2.7980
    Epoch   2  train_wf1=0.3032  val_wf1=0.1891  loss=2.6626
    Epoch   3  train_wf1=0.3756  val_wf1=0.3812  loss=2.6862
    Epoch  10  train_wf1=0.6125  val_wf1=0.5186  loss=2.2888
    Epoch  20  train_wf1=0.7390  val_wf1=0.5307  loss=2.4126
    Epoch  30  train_wf1=0.8099  val_wf1=0.5628  loss=2.4154
    Early stop at epoch 31

  Fold 1 done in 0.6 min
  WF1=0.5788  UAF1=0.5871  ACC=0.5868

Fold 2/5  |  Test: Session2
  Train: 121 dialogs  |  Val: 30 dialogs
  Params: 1,389,451
    Class counts: {'ang': 966, 'exc': 831, 'fru': 1524, 'hap': 478, 'neu': 1346, 'sad': 887}
    Epoch   1  train_wf1=0.2153  val_wf1=0.1328  loss=2.8985
    Epoch   2  train_wf1=0.3012  val_wf1=0.1689  loss=2.6661
    Epoch   3  train_wf1=0.3606  val_wf1=0.3182  loss=2

In [11]:
# ============================================================
#  Aggregate fold results → JSON
# ============================================================
wf1_list  = [fold_results[s]['metrics']['wf1']  for s in SESSIONS]
uaf1_list = [fold_results[s]['metrics']['uaf1'] for s in SESSIONS]
acc_list  = [fold_results[s]['metrics']['acc']  for s in SESSIONS]

summary = {
    'wf1_mean' : float(np.mean(wf1_list)),
    'wf1_std'  : float(np.std(wf1_list)),
    'uaf1_mean': float(np.mean(uaf1_list)),
    'uaf1_std' : float(np.std(uaf1_list)),
    'acc_mean' : float(np.mean(acc_list)),
    'acc_std'  : float(np.std(acc_list)),
    'per_fold' : {s: fold_results[s]['metrics'] for s in SESSIONS},
}

print('IEMOCAP 5-Fold LOSO — Plan A Results')
print(f'  WF1  : {summary["wf1_mean"]:.4f} +/- {summary["wf1_std"]:.4f}')
print(f'  UAF1 : {summary["uaf1_mean"]:.4f} +/- {summary["uaf1_std"]:.4f}')
print(f'  ACC  : {summary["acc_mean"]:.4f} +/- {summary["acc_std"]:.4f}')
print('\nPer-fold WF1:')
for s in SESSIONS:
    print(f'  {s}: {fold_results[s]["metrics"]["wf1"]:.4f}')

with open(RESULTS_DIR / 'final_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

full_save = {
    'summary'         : summary,
    'config'          : {'hidden': HIDDEN, 'k_layers': K_LAYERS,
                         'w_past': W_PAST, 'w_future': W_FUTURE,
                         'lambda_bcl': LAMBDA_BCL, 'lr': LR, 'epochs': EPOCHS},
    'per_fold_history': {s: fold_results[s]['history'] for s in SESSIONS},
}
with open(RESULTS_DIR / 'full_results.json', 'w') as f:
    json.dump(full_save, f, indent=2)

print(f'\nSaved: final_results.json, full_results.json -> {RESULTS_DIR}')

IEMOCAP 5-Fold LOSO — Plan A Results
  WF1  : 0.6165 +/- 0.0281
  UAF1 : 0.6051 +/- 0.0210
  ACC  : 0.6191 +/- 0.0293

Per-fold WF1:
  Session1: 0.5788
  Session2: 0.5921
  Session3: 0.6205
  Session4: 0.6358
  Session5: 0.6556

Saved: final_results.json, full_results.json -> /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/stage3_results


In [12]:
# ============================================================
#  Visualization 1: Training curves (loss + WF1 per fold)
# ============================================================
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for fi, sess in enumerate(SESSIONS):
    hist = fold_results[sess]['history']
    ep   = range(1, len(hist['train_loss']) + 1)

    ax = axes[0, fi]
    ax.plot(ep, hist['train_loss'], color='steelblue',  label='train', lw=1.5)
    ax.plot(ep, hist['val_loss'],   color='darkorange', label='val',   lw=1.5)
    ax.set_title(f'Fold {fi+1} ({sess})\nLoss', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)

    ax = axes[1, fi]
    ax.plot(ep, hist['train_wf1'], color='steelblue',  label='train', lw=1.5)
    ax.plot(ep, hist['val_wf1'],   color='darkorange', label='val',   lw=1.5)
    best_ep = int(np.argmax(hist['val_wf1'])) + 1
    ax.axvline(best_ep, color='red', ls=':', lw=1, alpha=0.7)
    best_v  = max(hist['val_wf1'])
    ax.set_title(f'WF1 (best={best_v:.4f})', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.legend(fontsize=7); ax.set_ylim(0, 1); ax.tick_params(labelsize=7)

plt.suptitle('IEMOCAP Plan A — Training Curves (5-Fold LOSO)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

Saved: training_curves.png


In [13]:
# ============================================================
#  Visualization 2: Confusion matrices (one per fold, normalized)
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(27, 5))

for fi, sess in enumerate(SESSIONS):
    true_l = fold_results[sess]['true']
    pred_l = fold_results[sess]['preds']
    cm     = confusion_matrix(true_l, pred_l, labels=list(range(NUM_CLASSES)))
    cm_n   = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    ax = axes[fi]
    sns.heatmap(cm_n, ax=ax,
                xticklabels=IEMOCAP_EMOTIONS, yticklabels=IEMOCAP_EMOTIONS,
                annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
                annot_kws={'size': 8}, linewidths=0.5)
    ax.set_title(f'Fold {fi+1}\n{sess}', fontsize=9)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('IEMOCAP Plan A — Normalized Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

Saved: confusion_matrices.png


In [14]:
# ============================================================
#  Visualization 3: Per-class F1 bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(11, 5))
x      = np.arange(NUM_CLASSES)
width  = 0.14
colors = plt.cm.tab10(np.linspace(0, 0.5, 5))

for fi, sess in enumerate(SESSIONS):
    per_cls = fold_results[sess]['metrics']['per_class']
    ax.bar(x + (fi - 2) * width, per_cls, width,
           label=f'Fold {fi+1}', color=colors[fi], alpha=0.85)

mean_cls = np.mean([fold_results[s]['metrics']['per_class'] for s in SESSIONS], axis=0)
ax.plot(x, mean_cls, 'k^--', label='Mean', lw=2, markersize=8, zorder=5)

for ci, mv in enumerate(mean_cls):
    ax.text(ci, mv + 0.015, f'{mv:.3f}', ha='center', va='bottom',
            fontsize=8, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(IEMOCAP_EMOTIONS, fontsize=11)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('IEMOCAP Plan A — Per-Class F1 (5-Fold LOSO)', fontsize=12)
ax.legend(ncol=3, fontsize=9); ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_class_f1.png')

Saved: per_class_f1.png


In [15]:
# ============================================================
#  Visualization 4: t-SNE of BCL head representations
# ============================================================
all_reps   = np.concatenate([fold_results[s]['reps']        for s in SESSIONS], axis=0)
all_labs   = np.concatenate([fold_results[s]['reps_labels'] for s in SESSIONS], axis=0)

# Subsample to 3000 points for speed
if len(all_reps) > 3000:
    idx      = np.random.choice(len(all_reps), 3000, replace=False)
    all_reps = all_reps[idx]
    all_labs = all_labs[idx]

print(f'Running t-SNE on {len(all_reps)} points (128-dim)...')
tsne = TSNE(n_components=2, perplexity=40, random_state=42,
            max_iter=1000, learning_rate='auto', init='pca')
X_2d = tsne.fit_transform(all_reps)
print('t-SNE done.')

palette = plt.cm.tab10(np.linspace(0, 0.6, NUM_CLASSES))
fig, ax = plt.subplots(figsize=(9, 7))
for ci, emo in enumerate(IEMOCAP_EMOTIONS):
    mask = all_labs == ci
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[palette[ci]], label=f'{emo} (n={mask.sum()})',
               alpha=0.55, s=12, linewidths=0)
ax.legend(markerscale=2.5, fontsize=10, loc='best', framealpha=0.85)
ax.set_title('IEMOCAP Plan A — t-SNE of Utterance Representations\n'
             '(BCL head, 128-dim, colored by emotion)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tsne_representations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: tsne_representations.png')

Running t-SNE on 3000 points (128-dim)...
t-SNE done.
Saved: tsne_representations.png


In [16]:
# ============================================================
#  Visualization 5: Visual arc attention heatmap
#  Shows which temporal segment (begin/mid/end) each emotion
#  relies on most — validates the expression arc hypothesis.
# ============================================================
arc_mean = np.zeros((NUM_CLASSES, 3))
arc_std  = np.zeros((NUM_CLASSES, 3))

for ci in range(NUM_CLASSES):
    all_w = []
    for sess in SESSIONS:
        segs = fold_results[sess]['arc_by_label'][ci]
        if segs:
            all_w.append(np.concatenate(segs, axis=0))
    if all_w:
        stacked     = np.concatenate(all_w, axis=0)
        arc_mean[ci] = stacked.mean(0)
        arc_std[ci]  = stacked.std(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap
im = axes[0].imshow(arc_mean, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['Begin', 'Mid', 'End'], fontsize=11)
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_yticklabels(IEMOCAP_EMOTIONS, fontsize=11)
axes[0].set_title('Mean Arc Attention Weight\nby Emotion Class', fontsize=11)
axes[0].set_xlabel('Temporal Segment')
axes[0].set_ylabel('Emotion')
for ci in range(NUM_CLASSES):
    for ti in range(3):
        clr = 'white' if arc_mean[ci, ti] > 0.55 else 'black'
        axes[0].text(ti, ci, f'{arc_mean[ci,ti]:.3f}',
                     ha='center', va='center', fontsize=9, color=clr)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Right: stacked bar per emotion
x  = np.arange(NUM_CLASSES)
w0 = arc_mean[:, 0]; w1 = arc_mean[:, 1]; w2 = arc_mean[:, 2]
axes[1].bar(x, w0,       label='Begin', color='#2196F3', alpha=0.9)
axes[1].bar(x, w1, w0,   label='Mid',   color='#FF9800', alpha=0.9)
axes[1].bar(x, w2, w0+w1, label='End',  color='#4CAF50', alpha=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(IEMOCAP_EMOTIONS, fontsize=10)
axes[1].set_ylabel('Attention Weight')
axes[1].set_ylim(0, 1)
axes[1].set_title('Arc Attention Distribution\nper Emotion Class', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('IEMOCAP Plan A — Visual Arc Attention (Expression Arc Analysis)',
             fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'arc_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: arc_attention_heatmap.png')
print('Interpretation: if begin/end weights differ by emotion, the arc carries signal.')
print('Uniform weights (~0.333) across segments suggest flat affect dominates.')

Saved: arc_attention_heatmap.png
Interpretation: if begin/end weights differ by emotion, the arc carries signal.
Uniform weights (~0.333) across segments suggest flat affect dominates.


In [17]:
# ============================================================
#  Visualization 6: Cross-fold performance summary
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(5)

bars_wf1  = ax.bar(x - 0.2, wf1_list,  0.35, label='WF1',  color='steelblue',  alpha=0.9)
bars_uaf1 = ax.bar(x + 0.2, uaf1_list, 0.35, label='UAF1', color='darkorange', alpha=0.9)

m_wf1  = float(np.mean(wf1_list))
m_uaf1 = float(np.mean(uaf1_list))
ax.axhline(m_wf1,  color='steelblue',  ls='--', lw=2, alpha=0.8,
           label=f'WF1 mean={m_wf1:.4f}')
ax.axhline(m_uaf1, color='darkorange', ls='--', lw=2, alpha=0.8,
           label=f'UAF1 mean={m_uaf1:.4f}')

for bi, (b1, b2) in enumerate(zip(bars_wf1, bars_uaf1)):
    ax.text(b1.get_x() + b1.get_width()/2, b1.get_height() + 0.004,
            f'{wf1_list[bi]:.4f}', ha='center', va='bottom', fontsize=7.5)
    ax.text(b2.get_x() + b2.get_width()/2, b2.get_height() + 0.004,
            f'{uaf1_list[bi]:.4f}', ha='center', va='bottom', fontsize=7.5)

fold_labels = [f'Fold {i+1}\n({s[:7]})' for i, s in enumerate(SESSIONS)]
ax.set_xticks(x); ax.set_xticklabels(fold_labels, fontsize=9)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title(
    f'IEMOCAP Plan A — Cross-Fold Performance\n'
    f'WF1: {m_wf1:.4f}+/-{float(np.std(wf1_list)):.4f}  '
    f'UAF1: {m_uaf1:.4f}+/-{float(np.std(uaf1_list)):.4f}',
    fontsize=11
)
y_max = max(max(wf1_list), max(uaf1_list)) + 0.08
ax.set_ylim(0, min(1.0, y_max))
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cross_fold_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cross_fold_summary.png')

Saved: cross_fold_summary.png


In [18]:
# ============================================================
#  Final confirmation
# ============================================================
print('=' * 60)
print('IEMOCAP Plan A — Complete')
print('=' * 60)
print(f'\nResults directory: {RESULTS_DIR}')
saved_files = sorted(RESULTS_DIR.iterdir())
for fp in saved_files:
    size_kb = fp.stat().st_size / 1024
    print(f'  {fp.name:<44} {size_kb:>8.1f} KB')

print(f'\nFinal WF1  : {summary["wf1_mean"]:.4f} +/- {summary["wf1_std"]:.4f}')
print(f'Final UAF1 : {summary["uaf1_mean"]:.4f} +/- {summary["uaf1_std"]:.4f}')
print(f'Final ACC  : {summary["acc_mean"]:.4f} +/- {summary["acc_std"]:.4f}')
print('\nComparison targets:')
print('  MM-DFN  baseline  WF1: ~58.18  (beat this first)')
print('  GraphSmile TPAMI  WF1: ~68-70  (aim for this range)')
print('  HRG-SSA IJCAI-25  WF1: ~75.47  (current ceiling)')

IEMOCAP Plan A — Complete

Results directory: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/stage3_results
  arc_attention_heatmap.png                        92.8 KB
  confusion_matrices.png                          164.7 KB
  cross_fold_summary.png                           60.9 KB
  final_results.json                                1.9 KB
  fold1_best_model.pt                            5444.8 KB
  fold2_best_model.pt                            5445.5 KB
  fold3_best_model.pt                            5445.2 KB
  fold4_best_model.pt                            5445.8 KB
  fold5_best_model.pt                            5445.0 KB
  full_results.json                                26.6 KB
  per_class_f1.png                                 59.6 KB
  training_curves.png                             310.6 KB
  tsne_representations.png                        244.6 KB

Final WF1  : 0.6165 +/- 0.0281
Final UAF1 : 0.6051 +/- 0.0210
Final ACC  : 0.6191 +/- 0.0293

Comparison targets